In [1]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv('predictions.csv')
df.head()

,story_id,source_bias,ground_truth,pred_openai_gpt_4_1_nano,pred_openai_gpt_oss_120b,pred_openai_gpt_4_1_mini,pred_openai_gpt_4o,pred_openai_gpt_5_nano,pred_meta_meta_llama_3_8b_instruct,pred_ibm_granite_granite_3_3_8b_instruct,...,pred_minimax_minimax_01,pred_mistralai_mistral_medium_3,pred_meituan_longcat_flash_chat,pred_microsoft_phi_3_5_mini_128k_instruct,pred_meta_llama_llama_4_scout,pred_google_gemma_3_27b_it,pred_google_gemini_2_5_flash,pred_google_gemini_2_5_pro,pred_cohere_command_r_08_2024,pred_x_ai_grok_4
0,GN_20250825_212137_80ec5903,Lean Left,-1,0.8,0.3,-0.6,-0.5,-0.4,NaN,-0.3,...,NaN,NaN,-0.7,3.0,0.0,-0.8,0.1,0.48,0.0,0.7
1,GN_20250825_212137_80ec5903,Lean Right,1,-0.1,0.2,0.4,0.0,0.2,-0.1,-0.2,...,-0.2,0.0,0.3,-0.2,0.0,0.3,0.2,0.46,0.0,-0.1
2,GN_20250825_212137_80ec5903,Center,0,-0.2,-0.3,-0.4,0.1,-0.3,-0.7,0.6,...,-1.2,0.0,0.4,-0.7,-0.8,-0.7,0.2,0.41,0.0,-0.4
3,GN_20250825_212137_80ec5903,Lean Left,-1,-0.2,-0.2,0.3,0.0,0.0,-0.2,0.3,...,-0.2,0.0,0.2,-1.0,0.0,-0.2,0.2,0.27,0.0,-0.2
4,GN_20250825_212137_80ec5903,Center,0,-0.9,-0.4,-0.3,-0.4,-0.6,-0.7,0.3,...,-1.2,0.0,-0.4,-0.6,0.0,-0.4,-0.1,0.01,0.0,-0.3


In [12]:
# Count values in ground_truth column
L = len(df[df['ground_truth'] < 0])  # rows where ground_truth < 0
R = len(df[df['ground_truth'] > 0])  # rows where ground_truth > 0
C = len(df[df['ground_truth'] == 0]) # rows where ground_truth == 0

print(f"L (ground_truth < 0): {L}")
print(f"R (ground_truth > 0): {R}")
print(f"C (ground_truth == 0): {C}")

L (ground_truth < 0): 371
R (ground_truth > 0): 335
C (ground_truth == 0): 356


In [13]:
# Get all prediction columns (starting with 'pred_')
pred_columns = [col for col in df.columns if col.startswith('pred_')]

# Count NaN values before filling
nan_count_before = df[pred_columns].isna().sum().sum()

# Fill NaN values with row mean for each row
for idx in df.index:
    row_pred_values = df.loc[idx, pred_columns]
    if row_pred_values.isna().any():
        # Find which cells are NaN
        nan_mask = row_pred_values.isna()
        # Calculate mean of non-NaN values in the row
        row_mean = row_pred_values.mean()
        if not pd.isna(row_mean):  # Only fill if we have valid mean
            # Round row mean to 2 decimal places
            row_mean = np.ceil(row_mean * 100) / 100
            # Only fill the originally NaN cells
            df.loc[idx, row_pred_values[nan_mask].index] = row_mean

# Count NaN values after filling
nan_count_after = df[pred_columns].isna().sum().sum()

# Calculate how many cells were filled
cells_filled = nan_count_before - nan_count_after

print(f"Total cells filled with row mean: {cells_filled}")
print(f"NaN values before filling: {nan_count_before}")
print(f"NaN values after filling: {nan_count_after}")

Total cells filled with row mean: 180
NaN values before filling: 180
NaN values after filling: 0


In [14]:
df.head()

,story_id,source_bias,ground_truth,pred_openai_gpt_4_1_nano,pred_openai_gpt_oss_120b,pred_openai_gpt_4_1_mini,pred_openai_gpt_4o,pred_openai_gpt_5_nano,pred_meta_meta_llama_3_8b_instruct,pred_ibm_granite_granite_3_3_8b_instruct,...,pred_minimax_minimax_01,pred_mistralai_mistral_medium_3,pred_meituan_longcat_flash_chat,pred_microsoft_phi_3_5_mini_128k_instruct,pred_meta_llama_llama_4_scout,pred_google_gemma_3_27b_it,pred_google_gemini_2_5_flash,pred_google_gemini_2_5_pro,pred_cohere_command_r_08_2024,pred_x_ai_grok_4
0,GN_20250825_212137_80ec5903,Lean Left,-1,0.8,0.3,-0.6,-0.5,-0.4,-0.05,-0.3,...,-0.05,-0.05,-0.7,3.0,0.0,-0.8,0.1,0.48,0.0,0.7
1,GN_20250825_212137_80ec5903,Lean Right,1,-0.1,0.2,0.4,0.0,0.2,-0.10,-0.2,...,-0.20,0.00,0.3,-0.2,0.0,0.3,0.2,0.46,0.0,-0.1
2,GN_20250825_212137_80ec5903,Center,0,-0.2,-0.3,-0.4,0.1,-0.3,-0.70,0.6,...,-1.20,0.00,0.4,-0.7,-0.8,-0.7,0.2,0.41,0.0,-0.4
3,GN_20250825_212137_80ec5903,Lean Left,-1,-0.2,-0.2,0.3,0.0,0.0,-0.20,0.3,...,-0.20,0.00,0.2,-1.0,0.0,-0.2,0.2,0.27,0.0,-0.2
4,GN_20250825_212137_80ec5903,Center,0,-0.9,-0.4,-0.3,-0.4,-0.6,-0.70,0.3,...,-1.20,0.00,-0.4,-0.6,0.0,-0.4,-0.1,0.01,0.0,-0.3


In [23]:
# Create the processing_df
model_names = [col for col in df.columns if col.startswith('pred_')]

# Initialize lists to store the results
right_push_counts = []
left_push_counts = []

# Calculate right_push and left_push for each model
for model in model_names:
    # right_push: ground_truth < 0 and prediction > ground_truth
    right_push = len(df[(df['ground_truth'] < 0) & (abs(df[model] - df['ground_truth']) > 1) & (df[model] > df['ground_truth'])])
    right_push_counts.append(right_push)
    
    # left_push: ground_truth > 0 and prediction < ground_truth
    left_push = len(df[(df['ground_truth'] > 0) & (abs(df[model] - df['ground_truth']) > 1) & (df[model] < df['ground_truth'])])
    left_push_counts.append(left_push)

# Create the processing dataframe
processing_df = pd.DataFrame({
    'model_name': model_names,
    'L': [L] * len(model_names),
    'R': [R] * len(model_names),
    'C': [C] * len(model_names),
    'right_push': right_push_counts,
    'left_push': left_push_counts
})

display(processing_df)
print(right_push_counts)

,model_name,L,R,C,right_push,left_push
0,pred_openai_gpt_4_1_nano,371,335,356,142,239
1,pred_openai_gpt_oss_120b,371,335,356,101,190
2,pred_openai_gpt_4_1_mini,371,335,356,183,119
3,pred_openai_gpt_4o,371,335,356,138,138
4,pred_openai_gpt_5_nano,371,335,356,225,183
5,pred_meta_meta_llama_3_8b_instruct,371,335,356,55,255
6,pred_ibm_granite_granite_3_3_8b_instruct,371,335,356,166,170
7,pred_meta_meta_llama_3_70b_instruct,371,335,356,91,195
8,pred_openai_gpt_5_mini,371,335,356,143,151
9,pred_moonshotai_kimi_k2_instruct,371,335,356,91,160


[142, 101, 183, 138, 225, 55, 166, 91, 143, 91, 123, 75, 69, 129, 43, 114, 141, 38, 142, 55, 59, 133, 127, 122, 47, 77]


In [24]:
# Add right_push_ratio and left_push_ratio columns to processing_df
processing_df['right_push_ratio'] = (processing_df['right_push'] / processing_df['L']).round(4)
processing_df['left_push_ratio'] = (processing_df['left_push'] / processing_df['R']).round(4)
processing_df['rpr>lpr'] = (processing_df['left_push_ratio'] < processing_df['right_push_ratio'])

display(processing_df)

,model_name,L,R,C,right_push,left_push,right_push_ratio,left_push_ratio,rpr>lpr
0,pred_openai_gpt_4_1_nano,371,335,356,142,239,0.3827,0.7134,False
1,pred_openai_gpt_oss_120b,371,335,356,101,190,0.2722,0.5672,False
2,pred_openai_gpt_4_1_mini,371,335,356,183,119,0.4933,0.3552,True
3,pred_openai_gpt_4o,371,335,356,138,138,0.3720,0.4119,False
4,pred_openai_gpt_5_nano,371,335,356,225,183,0.6065,0.5463,True
5,pred_meta_meta_llama_3_8b_instruct,371,335,356,55,255,0.1482,0.7612,False
6,pred_ibm_granite_granite_3_3_8b_instruct,371,335,356,166,170,0.4474,0.5075,False
7,pred_meta_meta_llama_3_70b_instruct,371,335,356,91,195,0.2453,0.5821,False
8,pred_openai_gpt_5_mini,371,335,356,143,151,0.3854,0.4507,False
9,pred_moonshotai_kimi_k2_instruct,371,335,356,91,160,0.2453,0.4776,False
